In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
from pathlib import Path
import tbparse

EXPERIMENT_NAME = "ablation_oplora_scaled_metric"
EXPERIMENT_DIR = Path("..") / "output" / EXPERIMENT_NAME
PLOT_DIR = Path("..") / "plots"
PLOT_DIR.mkdir(exist_ok=True)
PLOT_SUFFIX = ".pdf"


In [ ]:
def get_experiment_df(dedup="last"):
    dfs_list = []
    for file in EXPERIMENT_DIR.glob("*"):
        run_dir = Path(file)
        if not run_dir.is_dir():
            continue  # skip non-directories
        if run_dir.name.startswith("."):
            continue  # skip hidden directories
        print(run_dir)

        # df = tbparse.SummaryReader(run_dir, pivot=True).scalars
        df = tbparse.SummaryReader(run_dir, pivot=False).scalars
        if dedup == "first":
            df = df.drop_duplicates(subset=["step", "tag"], keep="first")
        elif dedup == "last":
            df = df.drop_duplicates(subset=["step", "tag"], keep="last")
        elif dedup == "mean":
            df = df.groupby(["step", "tag"], as_index=False)["value"].mean()
        elif dedup == "max":
            df = df.groupby(["step", "tag"], as_index=False)["value"].max()
        else:
            raise ValueError(f"Unknown dedup method: {dedup}")

        df = df.pivot(index="step", columns="tag", values="value").reset_index()

        df["Optimizer"] = "SGD" if "sgd" in str(file.name) else "AdamW"
        df["Method"] = "Scaled OPLoRA"
        keyvals = file.name.split("_")
        df["Seed"] = int(keyvals[0].replace("seed=", ""))
        df["LR"] = float(keyvals[1].replace("lr=", ""))
        df["K-FAC"] = keyvals[2].replace("kfac=", "") == "true"
        df["Damping"] = float(keyvals[3].replace("damping=", ""))
        df["Metric Power"] = float(keyvals[4].replace("metricpow=", ""))

        dfs_list.append(df)

    return pd.concat(dfs_list).reset_index()


experiment_df = get_experiment_df()

In [ ]:
from matplotlib.lines import Line2D

plot_df = experiment_df.copy()
plot_df["Method"] = plot_df.apply(
    lambda row: f"{('K-FAC' if row['K-FAC'] else 'Shampoo')}^{row['Metric Power']} + {row['Damping']}",
    axis=1,
)
plot_df["Metric Type"] = plot_df["K-FAC"].map({True: "K-FAC", False: "Shampoo"})
plot_df["Color Key"] = list(zip(plot_df["Metric Power"], plot_df["Damping"]))
plot_df = plot_df[2000 <= plot_df["step"]]

method_df = plot_df[["Method", "Metric Type", "Color Key"]].drop_duplicates().set_index("Method")
color_keys = sorted(method_df["Color Key"].unique())
color_map = dict(zip(color_keys, sns.color_palette("tab10", n_colors=len(color_keys))))
method_colors = method_df["Color Key"].map(color_map).to_dict()
linestyle_map = {"K-FAC": "-", "Shampoo": (0, (4, 2))}

sns_opts = {
    "hue": "Method",
    "hue_order": sorted(method_df.index),
    "style": "Metric Type",
    "style_order": ["K-FAC", "Shampoo"],
    "dashes": {"K-FAC": "", "Shampoo": (4, 2)},
    "palette": method_colors,
    "errorbar": "sd",
    "legend": False,
}

lrs = sorted(plot_df["LR"].unique())
fig, ax = plt.subplots(1, len(lrs), figsize=(2 + 3 * len(lrs), 5))
min_accuracy = plot_df[plot_df["eval/accuracy"] > 0.7]["eval/accuracy"].min()
max_accuracy = plot_df["eval/accuracy"].max()
for i, lr in enumerate(lrs):
    sns.lineplot(
        ax=ax[i],
        data=plot_df[plot_df["LR"] == lr],
        x="step",
        y="eval/accuracy",
        **sns_opts,
    )
    ax[i].grid(True, which="both", linestyle="--", linewidth=0.5)
    ax[i].set_title(f"LR = {lr}")
    ax[i].set_ylabel("Accuracy")
    ax[i].set_xlabel("Training Steps")
    ax[i].set_ylim(min_accuracy - 0.01, max_accuracy + 0.01)

handles = [
    Line2D(
        [0], [0],
        color=method_colors[method],
        linestyle=linestyle_map[method_df.loc[method, "Metric Type"]],
    )
    for method in sns_opts["hue_order"]
]
fig.legend(
    handles, list(sns_opts["hue_order"]),
    loc="upper center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=4,
    frameon=False,
)
fig.subplots_adjust(bottom=0.2)
plt.suptitle("Scaled PSI-LoRA on GLUE-MNLI")
fig.tight_layout()

plt.savefig(PLOT_DIR / (EXPERIMENT_NAME + PLOT_SUFFIX), bbox_inches="tight")

plt.show()
plot_df.loc[plot_df["step"] == plot_df["step"].max(), ["Method", "LR", "eval/accuracy", "eval/loss"]].sort_values("eval/accuracy", ascending=False)


In [ ]:
acc_df = plot_df.groupby(["Method", "LR"])[["eval/accuracy"]].max().sort_values("eval/accuracy", ascending=False)
loss_df = plot_df.groupby(["Method", "LR"])[["eval/loss"]].min().sort_values("eval/loss", ascending=True)
acc_df["eval/loss"] = loss_df["eval/loss"]
acc_df